# DiffInfinite - NIPA 병리 이미지 생성

학습된 DiffInfinite 모델로 이미지 10,000장 생성
- BR 클래스 (BRDC, BRID, BRIL, BRLC, BRNT): 각 1,000장 = 5,000장
- ST 클래스 (STDI, STIN, STMX, STNT): 각 1,250장 = 5,000장
- 저장 경로: `../../../result/NIPA/diffinfinite/`

## 1. 환경 설정 및 Import

In [ ]:
import os
import sys
import math
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from tqdm import tqdm
from ema_pytorch import EMA
from diffusers import AutoencoderKL

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from dm import Unet, GaussianDiffusion

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "CPU only")

## 2. 설정값 정의

In [ ]:
# 경로
MODEL_SAVE_DIR = "../../../model/NIPA/diffinfinite"
SAVE_DIR = "../../../result/NIPA/diffinfinite"
os.makedirs(SAVE_DIR, exist_ok=True)

# 체크포인트 번호 (가장 최신)
CHECKPOINT_MILESTONE = 46

# 모델 하이퍼파라미터 (학습 시 사용한 값과 동일)
IMAGE_SIZE = 1024
Z_SIZE = IMAGE_SIZE // 8   # 128
DIM = 256
DIM_MULTS = (1, 2, 4)
CHANNELS = 4
RESNET_BLOCK_GROUPS = 2
BLOCK_PER_LAYER = 2
TIMESTEPS = 1000
SAMPLING_TIMESTEPS = 250

# 클래스 정보 (sorted order)
CLASS_NAMES = ["BRDC", "BRID", "BRIL", "BRLC", "BRNT", "STDI", "STIN", "STMX", "STNT"]
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}

# 생성 설정
BR_CLASSES = ["BRDC", "BRID", "BRIL", "BRLC", "BRNT"]  # 5 classes x 1000 = 5000
ST_CLASSES = ["STDI", "STIN", "STMX", "STNT"]            # 4 classes x 1250 = 5000
BR_PER_CLASS = 1000
ST_PER_CLASS = 1250
BATCH_SIZE = 4  # 한 번에 생성할 이미지 수 (VRAM에 맞게 조절)
COND_SCALE = 3.0

# 클래스별 저장 폴더 생성
for cls_name in CLASS_NAMES:
    os.makedirs(os.path.join(SAVE_DIR, cls_name), exist_ok=True)

print(f"클래스: {CLASS_TO_IDX}")
print(f"BR: {BR_CLASSES} x {BR_PER_CLASS} = {len(BR_CLASSES) * BR_PER_CLASS}")
print(f"ST: {ST_CLASSES} x {ST_PER_CLASS} = {len(ST_CLASSES) * ST_PER_CLASS}")
print(f"총 생성: {len(BR_CLASSES) * BR_PER_CLASS + len(ST_CLASSES) * ST_PER_CLASS}장")
print(f"저장 경로: {SAVE_DIR}")

## 3. 모델 로드

In [ ]:
# UNet 생성
unet = Unet(
    dim=DIM,
    num_classes=NUM_CLASSES,
    dim_mults=DIM_MULTS,
    channels=CHANNELS,
    resnet_block_groups=RESNET_BLOCK_GROUPS,
    block_per_layer=BLOCK_PER_LAYER,
)

# GaussianDiffusion 모델
diffusion_model = GaussianDiffusion(
    unet,
    image_size=Z_SIZE,
    timesteps=TIMESTEPS,
    sampling_timesteps=SAMPLING_TIMESTEPS,
    loss_type='l2',
)

# EMA 래퍼
ema = EMA(diffusion_model, beta=0.995, update_every=10)

# 체크포인트 로드
ckpt_path = os.path.join(MODEL_SAVE_DIR, f"model-{CHECKPOINT_MILESTONE}.pt")
print(f"Loading checkpoint: {ckpt_path}")
data = torch.load(ckpt_path, map_location=device)

diffusion_model.load_state_dict(data['model'])
ema.load_state_dict(data['ema'])
print(f"Checkpoint loaded (step {data['step']})")

# EMA 모델 사용
ema_model = ema.ema_model
ema_model.eval()
ema_model = ema_model.to(device)

# VAE 로드
print("Loading VAE from stabilityai/sd-vae-ft-mse...")
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse")
vae.eval()
for param in vae.parameters():
    param.requires_grad = False
vae = vae.to(device)

print("모델 로드 완료!")

## 4. 이미지 생성 함수 정의

In [ ]:
@torch.no_grad()
def generate_and_save(ema_model, vae, class_name, class_idx, num_images, batch_size, save_dir, cond_scale=3.0):
    """특정 클래스의 이미지를 생성하고 개별 파일로 저장"""
    class_save_dir = os.path.join(save_dir, class_name)
    
    # 이미 생성된 이미지 수 확인 (이어서 생성 가능)
    existing = len([f for f in os.listdir(class_save_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])
    remaining = num_images - existing
    
    if remaining <= 0:
        print(f"  {class_name}: 이미 {existing}장 존재, 스킵")
        return existing
    
    print(f"  {class_name} (idx={class_idx}): {remaining}장 생성 시작 (기존 {existing}장)")
    
    generated = 0
    img_idx = existing
    
    pbar = tqdm(total=remaining, desc=f"  {class_name}")
    while generated < remaining:
        current_batch = min(batch_size, remaining - generated)
        
        # 클래스 마스크 생성 (1, IMAGE_SIZE, IMAGE_SIZE) - 클래스 레이블로 채움
        masks = torch.full(
            (current_batch, 1, IMAGE_SIZE, IMAGE_SIZE),
            class_idx, dtype=torch.long, device=device
        )
        
        # 더미 이미지 (inp_mask=None이므로 사용되지 않지만, device 참조용)
        dummy_images = torch.zeros(
            (current_batch, CHANNELS, Z_SIZE, Z_SIZE), device=device
        )
        
        # Latent space에서 샘플 생성
        z_sampled = ema_model.sample(dummy_images, masks) * 50
        
        # VAE 디코딩
        decoded = torch.clamp(vae.decode(z_sampled).sample, 0, 1)
        
        # 개별 이미지 저장
        for i in range(current_batch):
            img = decoded[i].cpu()
            img_pil = T.ToPILImage()(img)
            save_path = os.path.join(class_save_dir, f"{class_name}_{img_idx:05d}.png")
            img_pil.save(save_path)
            img_idx += 1
        
        generated += current_batch
        pbar.update(current_batch)
    
    pbar.close()
    total = existing + generated
    print(f"  {class_name}: 완료 (총 {total}장)")
    return total

print("생성 함수 정의 완료")

## 5. BR 클래스 이미지 생성 (5,000장)